# Single-channel beam fit at 156.25 MHz

**Question.** How well can one channel be fitted with spherical harmonics alone ($\ell_{\max}=8$), without bridging across frequency? The score table compares it with HFSS and with the reference multi-frequency PCA beam (v0007); their plots are in `channel_vs_hfss.ipynb`.

**Model.** $d_i \simeq \left|\sum_{\ell m}\mathbf c_{\ell m}\,Y_{\ell m}(\Omega_i)\cdot\mathbf e_i\right|^2$. The $\mathbf c_{\ell m}$ are complex Cartesian 3-vectors on real spherical harmonics, the same basis as the DPSS/PCA fits. $\mathbf e_i$ is the arm-0 transmitter vector projected transverse to the line of sight. There is no HFSS prior and no spectral basis. With one transmitter arm, only the field component along $\mathbf e_i$ is measured. The orthogonal transverse component, and so the beam's total normalization, is unconstrained, so no separate amplitude is fitted (see the polarization table). The overall phase is unobservable.

**Fixed inputs**, identical to the reference beam `derived/beam/empirical_raster_v0007`:
- The 07-17 20:26:00–21:28:40 UTC raster (phase C, box-air, RFANT) and the tooth − gap signal.
- **Pointing** from `marjum-2026-07/point_table_v2-beta.npz`: commanded azimuth and IMU elevation, plus v0007's table-to-beam azimuth offset and geometry. The evidence that commanded azimuth is correct is in `single_ch_beam_fit_debug.ipynb`.
- **Masks:** 89 reversal glitches, the gap-ratio dropout mask, the 56-sample opening el≈0 slew, and `derived/beam/raster_flags_v0001` (broadband dropouts, spikes and tooth spikes).
- **Stripes:** 12° of commanded azimuth for train/validation/test.

The fit starts from the HFSS 156.25 MHz field projected onto the harmonics and scaled to the data. The loss is unweighted least squares in counts. A training-stripe fit gives the held-out scores, and an all-sample fit gives the maps.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import healpy as hp
import pyarrow.parquet as pq

PLOT_BACKEND = os.environ.get('EIGSEP_NOTEBOOK_BACKEND', 'widget')
get_ipython().run_line_magic('matplotlib', PLOT_BACKEND)
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
os.environ.setdefault('JAX_ENABLE_X64', 'true')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'marjum-2026-07/data').is_dir())
CAMPAIGN = ROOT / 'marjum-2026-07'

import eigsep_data
from eigsep_data import AntennaResolutionPolicy, MetadataIndex
from eigsep_data.beam_sim import read_beam
from scipy.special import sph_harm_y
from scipy.optimize import minimize
from IPython.display import display
import hashlib
import jax
import jax.numpy as jnp
jax.config.update('jax_enable_x64', True)
eigsep_data.set_campaign_root(CAMPAIGN)

RASTER_UTC = ('2026-07-17 20:26:00', '2026-07-17 21:28:40')
FREQ_MHZ = 156.25            # an HFSS slice; every HFSS slice sits on an arm-1 comb tooth
GAP_OFFSET = 4

# Geometry and polarisation from beam_explorer.ipynb (branch B, roll mount).
AZ_SIGN, EL_SIGN = -1.0, -1.0
AZ0, EL0 = 177.0, 0.0        # deg
ALPHA = 45.0                 # transmitter polarisation, deg ccw from East

In [ ]:
t0, t1 = (pd.Timestamp(s, tz='UTC').timestamp() for s in RASTER_UTC)

# --- corrected pointing, joined to the correlator rows on time -------------
# point_table_v2-beta: commanded azimuth, IMU elevation (the default pointing).
with np.load(CAMPAIGN / 'point_table_v2-beta.npz') as z:
    POINT_PROVENANCE = json.loads(str(z['provenance']))
    pointing = pd.DataFrame({key: z[key] for key in ('t_utc_s', 'az_deg', 'el_deg', 'quality')})
assert pointing.t_utc_s.min() <= t0   # the table covers the raster start
pointing = pointing[(pointing.t_utc_s >= t0 - 60) & (pointing.t_utc_s <= t1 + 60)]
print('pointing: %s (%s), from %s' % (POINT_PROVENANCE['product'], POINT_PROVENANCE['version'],
                                      POINT_PROVENANCE['source_compact']))
policy = AntennaResolutionPolicy.load(CAMPAIGN / 'curation/antenna_resolution.json')
bundle = MetadataIndex(CAMPAIGN / 'data').select(time=(t0, t1)).load_bundle(
    antenna='box-air', missing='skip', resolution_policy=policy)
meta = bundle.meta.reset_index(drop=True).assign(t=lambda x: x.time_best.astype(float))
j = pd.merge_asof(meta.sort_values('t').reset_index(names='bundle_row'),
                  pointing.sort_values('t_utc_s')[['t_utc_s', 'az_deg', 'el_deg', 'quality']],
                  left_on='t', right_on='t_utc_s', direction='nearest', tolerance=0.35)
j = j[(j.quality == 'ok') & (j.rfswitch == 'RFANT')].reset_index(drop=True)
assert len(j) == 6990, len(j)   # 6901 after glitch removal

def reversal_glitches(el_deg, tol=15.0):
    """Isolated solved-elevation faults: a sample is good if at least 3 of its
    4 neighbours (+/-1, +/-2) lie within `tol` deg of it across the +/-180
    wrap. The sweeps move ~3 deg per sample, so real data always passes.
    At reversals the IMU readout drops single samples to ~0 or -60 deg while
    the motor is pegged at -180, and those have at most one close neighbour.
    """
    el = np.asarray(el_deg, float)
    close = np.zeros(el.size, int)
    for k in (-2, -1, 1, 2):
        d = np.abs((el - np.roll(el, k) + 180.0) % 360.0 - 180.0)
        edge = slice(0, -k) if k < 0 else slice(-k, None)   # neighbours that wrapped around the array
        d[edge] = 0.0
        close += d <= tol
    return close < 3


glitch = reversal_glitches(j.el_deg.to_numpy())
print('dropped %d elevation glitches' % glitch.sum())
j = j[~glitch].reset_index(drop=True)

T = j.t.to_numpy()
AZ = j.az_deg.to_numpy()
EL = j.el_deg.to_numpy()

# --- the channel -------------------------------------------------------------
BEAM_CART, _, HFSS_FREQS = read_beam()
s = int(np.argmin(np.abs(HFSS_FREQS - FREQ_MHZ)))
ch = int(round(HFSS_FREQS[s] / (250.0 / 1024)))
col = {int(round(f / (250.0 / 1024))): i for i, f in enumerate(bundle.freqs_mhz)}
rows = j.bundle_row.to_numpy()
tooth = bundle.data[rows, col[ch]].astype(float)
gap = 0.5 * (bundle.data[rows, col[ch - GAP_OFFSET]] + bundle.data[rows, col[ch + GAP_OFFSET]])
DATA = tooth - gap
# Whole-band dropouts (e.g. 21:18:00-21:18:35) take the gap down with the tooth.
ok = (gap / np.median(gap) > 1 / 1.5) & (gap / np.median(gap) < 1.5) & np.isfinite(DATA)
print('channel %d = %.4f MHz; %d samples, %d masked as dropouts'
      % (ch, HFSS_FREQS[s], len(DATA), (~ok).sum()))

In [ ]:
# --- v0007 reference models, masks, split and geometry -----------------------
PRODUCT = CAMPAIGN / 'derived/beam/empirical_raster_v0007'
provenance = json.loads((PRODUCT / 'provenance.json').read_text())
for relative_path, expected in provenance['artifact_sha256'].items():
    assert hashlib.sha256((PRODUCT / relative_path).read_bytes()).hexdigest() == expected, relative_path
with np.load(PRODUCT / 'pca/diagnostics.npz') as z:
    pca = {key: z[key] for key in z.files}
assert np.allclose(T, pca['t'], rtol=0, atol=1e-5) and np.allclose(AZ, pca['az_table'])
fidx = int(np.flatnonzero(pca['channels'] == ch)[0])
ARM = int(pca['arms'][fidx])
FIT_OK = pca['good'][fidx].astype(bool)            # dropout mask, slew and raster flags removed
assert np.array_equal(FIT_OK, ok & ~pca['el0_slew'].astype(bool) & ~pca['sample_flag'] & ~pca['tooth_flag'][fidx])
SPLIT = pca['split']                                # 12-deg stripes of commanded az: 0 train, 1 val, 2 test
AZ_OFFSET = float(pca['az_offset_deg'])             # table frame -> beam frame
HFSS_REF = pca['hfss'][fidx]                        # HFSS, one training-stripe gain, v0007 geometry
PCA_REF = pca['empirical_exported'][fidx]           # all-sample PCA lmax=8 fit (reference beam)
PCA_REF_HELDOUT = pca['empirical_heldout'][fidx]
params = pca['params']
print('channel %d (%.3f MHz), arm %d; az offset %+.1f deg; geometry %s' % (ch, HFSS_FREQS[s], ARM, AZ_OFFSET,
      {k: round(v, 3) for k, v in provenance['geometry'].items()}))
print('samples: %d fit (%d raster-flagged samples removed), split counts %s' % (
      FIT_OK.sum(), pca['sample_flag'].sum(), np.bincount(SPLIT[FIT_OK], minlength=3)))

In [ ]:
# --- forward model -------------------------------------------------------------
LMAX = 8
ant = np.asarray(json.loads((CAMPAIGN / 'curation/horizon_profiles.json').read_text())['antenna_enu_m'])
tx = np.asarray(json.loads((CAMPAIGN / 'curation/transmitter_position.json').read_text())['best_estimate_enu_m'])


def rotation(az_deg, el_deg):
    # topo = Rx(E) @ Rz(A) @ body: azimuth rolls about the boresight, elevation tips.
    A, E = np.deg2rad(az_deg), np.deg2rad(el_deg)
    cA, sA, cE, sE = np.cos(A), np.sin(A), np.cos(E), np.sin(E)
    z, o = np.zeros_like(cA), np.ones_like(cA)
    Rz = np.stack([np.stack([cA, -sA, z], -1), np.stack([sA, cA, z], -1), np.stack([z, z, o], -1)], -2)
    Rx = np.stack([np.stack([o, z, z], -1), np.stack([z, cE, -sE], -1), np.stack([z, sE, cE], -1)], -2)
    return Rx @ Rz


def harmonics(theta, phi, lmax):
    # Real orthonormal spherical harmonics, ordered as in the DPSS/PCA fits.
    columns = []
    for ell in range(lmax + 1):
        for m in range(-ell, ell + 1):
            y = sph_harm_y(ell, abs(m), theta, phi)
            columns.append(y.real if m == 0 else np.sqrt(2) * (y.imag if m < 0 else y.real))
    return np.column_stack(columns)


pa, paz, pel, peast, pnorth = params
direction = tx + np.array([peast, pnorth, 0.]) - ant
direction /= np.linalg.norm(direction)
rot = rotation(-(AZ + AZ_OFFSET) + AZ0 + paz, -EL + EL0 + pel)
body = np.einsum('nji,j->ni', rot, direction)
theta = np.arccos(np.clip(body[:, 2], -1, 1))
phi = np.mod(np.arctan2(body[:, 1], body[:, 0]), 2 * np.pi)
alpha = np.deg2rad(45 + pa + 90 * ARM)
e_tx = np.einsum('nji,j->ni', rot, [-np.sin(alpha), np.cos(alpha), 0.])
e_tx -= np.sum(e_tx * body, axis=1)[:, None] * body
Y = harmonics(theta, phi, LMAX)
DESIGN = np.concatenate([Y * e_tx[:, c, None] for c in range(3)], axis=1)   # (N, 3 nlm)

# HFSS 156.25 MHz field projected onto the same harmonics: the starting point.
NSIDE = hp.npix2nside(BEAM_CART.shape[-1])
pix_theta, pix_phi = hp.pix2ang(NSIDE, np.arange(BEAM_CART.shape[-1]))
PIX_DIR = np.array(hp.pix2vec(NSIDE, np.arange(BEAM_CART.shape[-1])))
Y_PIX = harmonics(pix_theta, pix_phi, LMAX)
C_HFSS = np.linalg.lstsq(Y_PIX, BEAM_CART[s].T, rcond=None)[0].T.ravel()    # component-blocked
print('HFSS field truncation error at lmax=%d: %.2f%%' % (LMAX, 100 * np.linalg.norm(
    BEAM_CART[s] - C_HFSS.reshape(3, -1) @ Y_PIX.T) / np.linalg.norm(BEAM_CART[s])))


def unit_power_gain(coeff):
    # Mean spherical transverse power of the field: the gain on the unit-average-power beam.
    field = coeff.reshape(3, -1) @ Y_PIX.T
    field -= np.sum(field * PIX_DIR, axis=0) * PIX_DIR
    return float(np.mean(np.sum(abs(field) ** 2, axis=0)))

In [ ]:
# --- fit: harmonic coefficients only, unweighted least squares in counts -----
def fit_channel(use, maxiter=5000, design=None):
    design = DESIGN if design is None else design
    p0 = abs(design @ C_HFSS) ** 2
    c0 = C_HFSS * np.sqrt(np.sum(p0[use] * DATA[use]) / np.sum(p0[use] ** 2))
    scale = np.sqrt(np.mean(DATA[use] ** 2))
    D, target = jnp.asarray(design[use]), jnp.asarray(DATA[use] / scale)
    n = c0.size

    def objective(x):
        return jnp.mean((jnp.abs(D @ (x[:n] + 1j * x[n:])) ** 2 - target) ** 2)

    vg = jax.jit(jax.value_and_grad(objective))
    x0 = np.r_[c0.real, c0.imag] / np.sqrt(scale)
    opt = minimize(lambda x: tuple(np.asarray(v, float) for v in vg(jnp.asarray(x))),
                   x0, jac=True, method='L-BFGS-B',
                   options=dict(maxiter=maxiter, ftol=1e-12, gtol=1e-8, maxcor=30))
    coeff = (opt.x[:n] + 1j * opt.x[n:]) * np.sqrt(scale)
    return abs(design @ coeff) ** 2, coeff, opt


def frac_rms(model, use):
    return np.linalg.norm((DATA - model)[use]) / np.linalg.norm(DATA[use])


TRAIN = FIT_OK & (SPLIT == 0)
SINGLE_HELDOUT, _, train_opt = fit_channel(TRAIN)
SINGLE, single_coeff, all_opt = fit_channel(FIT_OK)
print('real parameters %d; train fit: %d iterations, %s; all-sample fit: %d iterations, %s'
      % (2 * single_coeff.size, train_opt.nit, train_opt.message, all_opt.nit, all_opt.message))

rows = []
for name, model, held in [('HFSS (one gain)', HFSS_REF, HFSS_REF),
                          ('v0007 PCA lmax=8, 42 channels', PCA_REF, PCA_REF_HELDOUT),
                          ('single-channel lmax=8', SINGLE, SINGLE_HELDOUT)]:
    rows.append(dict(model=name, train=frac_rms(held, TRAIN),
                     validation_heldout=frac_rms(held, FIT_OK & (SPLIT == 1)),
                     test_heldout=frac_rms(held, FIT_OK & (SPLIT == 2)),
                     all_samples=frac_rms(model, FIT_OK)))
display(pd.DataFrame(rows).set_index('model').round(4))

# One transmitter arm constrains only E.e. The transverse component orthogonal to
# e never reaches the data, so an unregularized fit lets it drift; the total beam
# normalization (and so a separate amplitude) is then not identified.
def pol_power(coeff):
    E = (coeff.reshape(3, -1) @ Y.T).T[FIT_OK]
    b = body[FIT_OK]
    E -= np.sum(E * b, axis=1)[:, None] * b
    e_hat = e_tx[FIT_OK] / np.linalg.norm(e_tx[FIT_OK], axis=1)[:, None]
    return (np.mean(abs(np.sum(E * e_hat, axis=1)) ** 2),
            np.mean(abs(np.sum(E * np.cross(b, e_hat), axis=1)) ** 2))


p0 = abs(DESIGN @ C_HFSS) ** 2
start = C_HFSS * np.sqrt(np.sum(p0[FIT_OK] * DATA[FIT_OK]) / np.sum(p0[FIT_OK] ** 2))
display(pd.DataFrame([dict(field=name, co_pol=co, cross_pol=cross, cross_over_co=cross / co)
                      for name, (co, cross) in [('HFSS start', pol_power(start)),
                                                ('single-channel fit', pol_power(single_coeff))]]
                     ).set_index('field').round(3))

In [ ]:
plt.rcParams.update({'font.size': 14, 'axes.spines.top': False, 'axes.spines.right': False})
az_plot = (AZ + 180.0) % 360.0 - 180.0          # the raster crosses 0/360; keep it contiguous
time = pd.to_datetime(T, unit='s', utc=True)

# Shared color scales for both models: log data/model, symmetric residual set by HFSS.
lo, hi = np.percentile(DATA[FIT_OK & (DATA > 0)], [1, 99.5])
norm = LogNorm(vmin=lo, vmax=hi)
rlim = np.percentile(np.abs((DATA - HFSS_REF)[FIT_OK]), 98)


def plot_maps(model, name, az=None):
    x = az_plot if az is None else (az + 180.0) % 360.0 - 180.0
    fig, axes = plt.subplots(1, 3, figsize=(19, 7), sharex=True, sharey=True,
                             constrained_layout=True)
    # LogNorm drops values <= 0, and tooth - gap is slightly negative in some nulls:
    # clip the log panels at the color floor so data and model show the same samples.
    panels = [('Data (tooth − gap)', np.clip(DATA, lo, None), dict(cmap='viridis', norm=norm)),
              (name + ' model', np.clip(model, lo, None), dict(cmap='viridis', norm=norm)),
              ('Residual (data − model)', DATA - model, dict(cmap='RdBu_r', vmin=-rlim, vmax=rlim))]
    for ax, (title, value, kw) in zip(axes, panels):
        sc = ax.scatter(x[FIT_OK], EL[FIT_OK], c=value[FIT_OK], s=4, **kw)
        ax.set_title(title)
        ax.set_xlabel('Azimuth [deg]')
        fig.colorbar(sc, ax=ax, orientation='horizontal', pad=0.1, shrink=0.85, label='Counts')
    axes[0].set_ylabel('Elevation [deg]')
    fig.suptitle('%s, %.3f MHz, raster %s–%s UTC'
                 % (name, HFSS_FREQS[s], RASTER_UTC[0][-8:], RASTER_UTC[1][-8:]), fontsize=17)
    plt.show()


def plot_time(model, name):
    fig, ax = plt.subplots(figsize=(16, 6.5), constrained_layout=True)
    ax.plot(time[FIT_OK], DATA[FIT_OK], '.', ms=2, color='0.25', label='data')
    ax.plot(time[FIT_OK], model[FIT_OK], '.', ms=2, color='tab:red', label=name + ' model')
    ax.plot(time[FIT_OK], (DATA - model)[FIT_OK], '.', ms=2, color='tab:blue', label='residual')
    ax.axhline(0, color='0.6', lw=1)
    ax.set_ylabel('Counts')
    ax.set_xlabel('Time [UTC]')
    ax.set_title('%s, %.3f MHz through the raster' % (name, HFSS_FREQS[s]))
    ax.legend(loc='upper right', markerscale=6)
    plt.show()

## Single-channel fit, $\ell_{\max}=8$

The data/model/residual maps and time series for HFSS and the v0007 PCA beam are in `channel_vs_hfss.ipynb`; this notebook uses the same samples and color scales.

In [ ]:
plot_maps(SINGLE, 'single-channel ℓmax=8')
plot_time(SINGLE, 'single-channel ℓmax=8')

## Findings

- **With commanded pointing and flagged samples removed, all three models sit at 5–7% fractional RMS at 156.25 MHz.**
  - HFSS with one gain is best on the validation stripes (0.059, against 0.069 for the v0007 PCA beam and 0.068 for this fit).
  - This single-channel fit is best on the test stripes (0.052, against 0.069 for HFSS and 0.068 for PCA).
  - The v0007 PCA beam is not better than HFSS at this frequency; its advantage is across the band.
  - The residual structure left in the maps is largely common to all three (compare with the HFSS and PCA maps in `channel_vs_hfss.ipynb`).
- **The single-channel field is not a usable full beam.** One arm constrains only the co-polar coupling; its cross-polar component drifts to 21× the co-polar power. Both fits also stop at the 5,000-iteration cap. The multi-frequency fit, with both arms and the HFSS prior, is the beam to use.
- **Caveats:** the stripes are inherited, not a fresh blind test; geometry is fixed at v0007's; the loss is unweighted in counts.